# Top 2 products sold per category


In [0]:
%python
data = [
    ("Electronics", "TV", 120),
    ("Electronics", "Laptop", 180),
    ("Electronics", "Phone", 200),
    ("Groceries", "Milk", 50),
    ("Groceries", "Eggs", 80),
    ("Groceries", "Bread", 70)
]

columns = ["category", "product", "sales"]

df = spark.createDataFrame(data, columns)

In [0]:
%python
from pyspark.sql.window import Window
from pyspark.sql.functions import *

window_spec = Window.partitionBy("category").orderBy(col("sales").desc())

top2 = df.withColumn("rn", row_number().over(window_spec)) \
         .filter("rn <= 2") \
         .drop("rn")

top2.show()

#Remove duplicates keeping latest timestamp


When we import Window from pyspark.sql.window, we are accessing the Window class, which provides methods such as partitionBy(), orderBy(), rowsBetween(), and rangeBetween() to create a WindowSpec object.

In [0]:
data = [
    (1, "A", "2024-01-01"),
    (1, "A", "2024-02-01"),
    (2, "B", "2024-01-05"),
    (2, "B", "2024-01-10"),
    (3, "C", "2024-03-01")
]

columns = ["id", "name", "timestamp"]
df=spark.createDataFrame(data,columns)


In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.window import Window

window_s = Window.partitionBy("id").orderBy(col("timestamp").desc())

df_latest = df.withColumn("rn", row_number().over(window_s)) \
              .filter("rn = 1") \
              .drop("rn")

display(df_latest)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

data = [
    (1, "2024-01-01"),
    (1, "2024-01-02"),
    (1, "2024-01-05"),
    (2, "2024-01-03"),
    (2, "2024-01-05")
]

columns = ["customer_id", "purchase_date"]

df = spark.createDataFrame(data, columns) \
          .withColumn("purchase_date", col("purchase_date").cast("date"))


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Define window
window_s = Window.partitionBy("customer_id") .orderBy(col("purchase_date"))

# Add previous purchase date
df_with_lag = df.withColumn("prev_date",lag("purchase_date").over(window_s))

# Calculate date difference and filter consecutive days
consecutive_df = df_with_lag.withColumn("date_diff",datediff(col("purchase_date"), col("prev_date"))).\
filter(col("date_diff") == 1)

display(consecutive_df)

#2nd Highest Salary per Department

In [0]:
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder.appName("SecondHighestSalary").getOrCreate()

# Sample data
data = [
    (1, "A", "HR", 5000),
    (2, "B", "HR", 7000),
    (3, "C", "HR", 6000),
    (4, "D", "IT", 9000),
    (5, "E", "IT", 8000),
    (6, "F", "IT", 9000)   # duplicate highest salary
]

columns = ["emp_id", "name", "department", "salary"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

df.show()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

Window_s = Window.partitionBy("Department").orderBy(col("salary").desc())

df_rank = df.withColumn("rn", row_number().over(Window_s)).filter(col("rn") == 2).drop("rn")

df_rank.show()

#Detect late deliveries


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.getOrCreate()

data = [
    (101, "2024-01-01", "2024-01-03"),  # Late
    (102, "2024-01-05", "2024-01-05"),  # On time
    (103, "2024-01-07", "2024-01-06"),  # Early
    (104, "2024-01-10", "2024-01-15")   # Late
]

columns = ["order_id", "expected_date", "actual_date"]

df = spark.createDataFrame(data, columns) \
          .withColumn("expected_date", col("expected_date").cast("date")) \
          .withColumn("actual_date", col("actual_date").cast("date"))

df.show()

In [0]:
df=df.filter(col("expected_date")<col("actual_date"))
display(df)

#Users inactive for last 30 days


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Create Spark session
spark = SparkSession.builder.appName("InactiveUsers").getOrCreate()

# Sample data: user_id, last_activity_date
data = [
    (1, "2025-01-01"),
    (2, "2025-02-15"),
    (3, "2025-02-20"),
    (4, "2025-01-25"),
    (5, "2026-02-28")
]

columns = ["user_id", "last_activity_date"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Convert last_activity_date to date type
df = df.withColumn("last_activity_date", col("last_activity_date").cast("date"))

df.show()

In [0]:
from pyspark.sql.functions import current_date, datediff

inactive_users = df.filter(datediff(current_date(), col("last_activity_date")) > 30)

inactive_users.show()

#Daily active users aggregation--means daily how many people logged in


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("DAU").getOrCreate()

data = [
    (1, "2025-02-28", "login"),
    (2, "2025-02-28", "purchase"),
    (1, "2025-03-01", "login"),
    (3, "2025-03-01", "login"),
    (2, "2025-03-01", "login")
]

columns = ["user_id", "activity_date", "action"]

df = spark.createDataFrame(data, columns) \
          .withColumn("activity_date", col("activity_date").cast("date"))

df.show()

In [0]:
from pyspark.sql.functions import countDistinct, col

df_log = df.groupBy("activity_date") \
           .agg(countDistinct("user_id").alias("daily_active_users")) \
           .orderBy("activity_date")

display(df_log)

#Optimize join when one table is small



In [0]:
from pyspark.sql.functions import broadcast

df_result = df_large.join(
    broadcast(df_small),
    on="id",
    how="inner"
)